In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita visualização gráfica inline no Jupyter notebook
%matplotlib inline

# Transferir um decodificador entre sessões gravadas

Treine em uma sessão de aquisição e faça predições em outra sessão para a mesma pessoa.
Carregamos o sujeito 1, sessões ``0train`` e ``1train``, execução 0 de imaginação motora
real de mão esquerda/direita de [NEMAR nm000135](https://nemar.org/dataset/nm000135)
(BNCI2014-004). Cada arquivo de sinal possui aproximadamente 5.5 MB; a primeira execução
requer conexão à internet. Defina ``EEGDASH_CACHE_DIR`` para reutilizar as gravações em CI.
O subconjunto disponível do catálogo suporta uma demonstração de sessão para um participante,
não uma reivindicação de generalização populacional.

Pré-requisitos: a distinção entre divisões por ensaios e por grupos do tutorial 11,
e o pipeline escalonador/classificador do tutorial 12. Execute esta página de forma independente
com EEGDash, Braindecode e scikit-learn instalados. Nomes de sessões são identidades de aquisição BIDS.
O sufixo ``train`` pertence ao lançamento de origem; ele não nos impede de reservar a segunda
sessão para avaliação.


## 1. Carregar dois identificadores genuínos de sessão



In [ ]:
# Importa módulos de sistema operacional, funções parciais e caminhos de arquivos
import os
from functools import partial
from pathlib import Path

# Importa bibliotecas para plotagem, matrizes numéricas e manipulação de tabelas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Importa gerador de janelas baseadas em eventos da Braindecode
from braindecode.preprocessing import create_windows_from_events
# Importa modelo de regressão logística e métricas do scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Importa classes de dataset e extratores de características do EEGDash
from eegdash import EEGDashDataset
from eegdash.features import (
    FeatureExtractor,
    extract_features,
    spectral_bands_power,
    spectral_preprocessor,
)

# Lista com os identificadores das duas sessões a serem carregadas
sessions = ["0train", "1train"]
# Inicializa e carrega as duas sessões de imaginação motora do sujeito 1
dataset = EEGDashDataset(
    cache_dir=Path(os.environ.get("EEGDASH_CACHE_DIR", ".eegdash_cache")),
    dataset="nm000135",
    subject="1",
    session=sessions,
    run="0",
    task="imagery",
    n_jobs=1,
)
# Valida que exatamente duas gravações (uma por sessão) foram carregadas
assert len(dataset.datasets) == 2
# Exibe metadados das gravações
print(dataset.description[["subject", "session", "run"]])

## 2. Inspecionar rótulos observados de mãos e selecionar canais de EEG
A versão foi convertida através do MOABB. Preserve seu pré-processamento e
a temporização dos eventos; selecionar EEG exclui quaisquer canais que não sejam de EEG das características.



In [ ]:
# Dicionário de mapeamento das condições de imaginação motora para rótulos binários (0 e 1)
mapping = {"left_hand": 0, "right_hand": 1}
# Itera pelas sessões para filtrar canais EEG e verificar consistência de anotações
for recording in dataset.datasets:
    raw = recording.raw
    # Seleciona estritamente os canais de EEG, removendo outros tipos de sensores
    raw.pick("eeg")
    # Assegura que os eventos mapeados existem nas anotações da gravação
    assert set(mapping).issubset(raw.annotations.description)
    # Exibe informações de sessão, canais, taxa de amostragem e eventos únicos presentes
    print(
        recording.description["session"],
        raw.ch_names,
        raw.info["sfreq"],
        np.unique(raw.annotations.description),
    )
# Extrai a frequência de amostragem e os nomes dos canais da primeira sessão como referência
sfreq = dataset.datasets[0].raw.info["sfreq"]
channels = dataset.datasets[0].raw.ch_names
# Assegura que todas as gravações possuem a mesma taxa de amostragem e mesmos canais
assert all(
    r.raw.ch_names == channels and r.raw.info["sfreq"] == sfreq
    for r in dataset.datasets
)

## 3. Criar uma janela de três segundos por ensaio real de imaginação
Os canais selecionados são C3, Cz e C4 sobre o córtex motor. A 250 Hz,
três segundos correspondem a 750 amostras; uma janela é (3 canais, 750 amostras)
em volts. As duas sessões produzem 120 ensaios rotulados cada neste subconjunto.
Manter uma janela por pista visual (*cue*) evita contar recortes sobrepostos como ensaios independentes.
Anotações BAD_ACQ_SKIP não são rótulos de classe; o mapeamento explícito
seleciona apenas pistas observadas de imaginação motora de mãos.



In [ ]:
# Define o tamanho da janela em amostras correspondente a 3 segundos (3 * 250 = 750 amostras)
window_size = int(3 * sfreq)
# Cria as janelas a partir dos eventos descartando fragmentos restantes ao final
windows = create_windows_from_events(
    dataset,
    mapping=mapping,
    trial_start_offset_samples=0,
    trial_stop_offset_samples=0,
    window_size_samples=window_size,
    window_stride_samples=window_size,
    on_last_window="drop",
    preload=True,
)
# Obtém a tabela de metadados das janelas criadas
metadata = windows.get_metadata()
# Garante que há exatamente uma janela extraída por ensaio
assert (metadata.i_window_in_trial == 0).all(), "Expected one window per trial"
# Garante unicidade dos ensaios evitando duplicação
assert not metadata.duplicated(["subject", "session", "run", "i_start_in_trial"]).any()
# Empilha os dados em uma matriz 3D (n_ensaios, n_canais, n_amostras)
X = np.stack([window[0] for window in windows])
# Extrai o array numérico das classes alvo y (0 ou 1)
y = metadata.target.to_numpy(dtype=int)
# Extrai o agrupador de sessões ('0train' ou '1train')
groups = metadata.session.astype(str).to_numpy()
# Valida presença correta das duas sessões e integridade dos números em X
assert set(groups) == set(sessions) and np.isfinite(X).all()
# Exibe as dimensões da matriz de janelas e a contagem cruzada por sessão e classe
print("Windows:", X.shape)
print(pd.crosstab(groups, y))

## 4. Extrair o log da potência na banda motora independentemente por ensaio
O EEGDash calcula um espectro Welch compartilhado e, em seguida, soma os bins de PSD nas
bandas de 8–13 Hz e 13–30 Hz separadamente para C3, Cz e C4. Isso gera seis
características por ensaio. A função pública de banda utiliza intervalos semiabertos, portanto
o bin de 13 Hz pertence apenas à banda beta. A multiplicação pelo espaçamento de bins
de 1/3 Hz aproxima a potência integrada em V² antes da aplicação do logaritmo.
A versão processada de imaginação motora não passa por uma segunda limpeza com EEGPrep.



In [ ]:
# Dicionário com a definição das bandas sensório-motoras: mu (8-13 Hz) e beta (13-30 Hz)
bands = {"mu": (8, 13), "beta": (13, 30)}
# Configura o extrator com pré-processador espectral de 8 a 30 Hz com segmentos de 750 amostras
spectral = FeatureExtractor(
    {"power": partial(spectral_bands_power, bands=bands)},
    preprocessor=partial(
        spectral_preprocessor,
        fs=sfreq,
        nperseg=window_size,
        noverlap=0,
        f_min=8,
        f_max=30,
    ),
)
# Executa a extração em lote para todas as janelas geradas
feature_table = extract_features(
    windows, {"spectral": spectral}, batch_size=64, n_jobs=1
).to_dataframe()
# Valida o número de colunas geradas (3 canais * 2 bandas = 6 características)
assert feature_table.shape == (len(y), len(channels) * len(bands))
# Converte para potência em V² multiplicando pelo bin width (sfreq / window_size = 1/3 Hz) e calcula o log natural
features = np.log(np.maximum(feature_table.to_numpy() * sfreq / window_size, 1e-30))
# Garante ausência de NaNs ou infinitos na matriz final de características
assert np.isfinite(features).all()

## 5. Transferência em ambas as direções com escalonamento apenas no treino
Treinar na sessão posterior é um diagnóstico retrospectivo. Apenas a direção
0train para 1train representa uma transferência prospectiva de sessão.



In [ ]:
# Inicializa lista para registrar as métricas de transferência entre as sessões
rows = []
# Avalia as duas direções de transferência: 0train -> 1train e 1train -> 0train
for train_session, test_session in [sessions, sessions[::-1]]:
    # Localiza índices dos ensaios de treino e teste baseados no nome da sessão
    train = np.flatnonzero(groups == train_session)
    test = np.flatnonzero(groups == test_session)
    # Assegura que não há sobreposição de amostras entre treino e teste
    assert set(groups[train]).isdisjoint(groups[test])
    # Assegura presença das duas classes em ambos os conjuntos
    assert set(y[train]) == set(y[test]) == set(mapping.values())
    # Cria pipeline com padronizador StandardScaler e classificador LogisticRegression
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    # Ajusta o pipeline estritamente nos dados da sessão de treino
    model.fit(features[train], y[train])
    # Gera predições para os ensaios da sessão de teste retida
    prediction = model.predict(features[test])
    # Registra o sentido da transferência, a acurácia balanceada e o número de ensaios de teste
    rows.append(
        dict(
            transfer=f"{train_session} → {test_session}",
            balanced_accuracy=balanced_accuracy_score(y[test], prediction),
            n_test=len(test),
        )
    )
# Constrói e exibe a tabela de resultados comparativos
results = pd.DataFrame(rows)
print(results.to_string(index=False))

## 6. Plotar pontuações medidas de transferência de sessão



In [ ]:
# Plota gráfico de barras comparando as pontuações de acurácia nas duas direções
results.plot.bar(x="transfer", y="balanced_accuracy", legend=False, rot=0)
# Linha horizontal pontilhada indicando o nível do acaso para classificação binária (50%)
plt.axhline(0.5, color="black", linestyle="--", label="Chance")
# Configura limites e título do eixo Y
plt.ylim(0, 1)
plt.ylabel("Balanced accuracy")
plt.legend()
# Exibe a figura
plt.show()

## 7. Decidir o que o resultado da transferência suporta
A acurácia balanceada é a média da sensibilidade de mão esquerda/direita, com o acaso em 0.5. A coluna
n_test impressa é o número de ensaios reais da sessão reservada. Diferenças
entre as direções podem refletir dificuldade no treinamento ou condições da sessão;
elas não isolam um mecanismo específico de desvio de eletrodos (*electrode drift*).

Para implantação após calibração, reserve uma sessão genuína posterior e faça ajustes
apenas dentro de sessões anteriores. Se você adicionar ensaios de calibração da sessão-alvo,
exclua esses ensaios do conjunto de teste dela e reporte quantos rótulos a adaptação utiliza.
Esse é um protocolo diferente da transferência com calibração zero realizada aqui.

